# Quality Checks — Breast Cancer Wisconsin (Diagnostic)

**Course:** UE 5BM753 — Supervised and Unsupervised Learning for Biology
**Part of:** the Breast Cancer Wisconsin (Diagnostic) final report, split into
per-step notebooks. See `00_Final_Report_.ipynb` for the combined narrative and
`README.md` for the overall conclusion.

**Question addressed here:** is the data trustworthy enough to model?

This notebook loads the raw dataset and checks it: missing values, duplicated rows, and outliers.

## Setup

Defines the paths used below and imports the reusable functions kept in
`01_scripts/`.

In [ ]:
import os
import sys

sys.path.append(os.path.join("..", "01_scripts"))
from dataset import load_data, get_feature_columns, check_missing_values, check_duplicates, detect_outliers_iqr

DATA_PATH = os.path.join("..", "02_data", "raw", "10_Breast_Cancer_Wisconsin_Diagnostic.csv")
RESULTS_DIR = os.path.join("..", "04_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

## Data

This project uses the **Breast Cancer Wisconsin (Diagnostic)** dataset. It contains 569
tumor samples. Each sample is described by 30 continuous measurements taken from a
digitized image of a fine-needle aspirate (a small tissue sample) of a breast mass.
Each sample is labeled as **benign** or **malignant**, based on an actual clinical
diagnosis.

The measurements describe the shape and texture of the cell nuclei visible in the
image. Malignant (cancerous) cells tend to look different from benign ones. They are
often larger, more irregular in shape, and rougher in texture. These measurements
should let us tell the two groups apart.

### Variables

Each sample is described by 30 continuous features. These features are derived from 10
base measurements of the cell nuclei visible in the image. Each base measurement is
summarized by three statistics: **mean**, **standard error (SE)**, and **worst**. The
"worst" value is the average of the 3 most extreme measurements, capturing the most
abnormal-looking nuclei in the sample. Together, these give the 30 features used
throughout this report, for example `radius_mean`, `radius_se`, and `radius_worst`.

| Feature | What it measures |
| --- | --- |
| `radius` | Mean distance from the center of the nucleus to its perimeter |
| `texture` | Standard deviation of gray-scale values. Rougher surfaces give higher values |
| `perimeter` | Total length of the nucleus's outer contour |
| `area` | Total 2D surface occupied by the nucleus |
| `smoothness` | Local variation in radius length. Measures edge regularity |
| `compactness` | perimeter^2 / area - 1.0. Measures how compact or round the shape is |
| `concavity` | Severity of the concave, indented portions of the contour |
| `concave points` | Number of concave portions of the contour |
| `symmetry` | How symmetric the nucleus shape is |
| `fractal dimension` | Structural complexity and irregularity of the contour |

### Quality checks

Before any modeling, we need to make sure the data is trustworthy. Below, the raw data
is loaded and previewed. Then three standard checks are run: missing values, duplicated rows, and outliers. Outliers are detected using the 1.5x IQR rule, a standard statistical cutoff.

Outliers are flagged for reporting only. They are not removed. In a biological dataset
like this one, an unusually large or irregular nucleus may simply reflect a more
aggressive tumor, not a measurement error. Deciding whether such a value is wrong or
biologically real requires domain knowledge, not an automatic rule. So we keep it and
note it instead.

In [ ]:
df = load_data(DATA_PATH) # drop id colum and encodes 'diagnosis' as a binary label (1 = malignant, 0 = benign)
features = get_feature_columns(df)

print(f"Dataset shape: {df.shape[0]} samples x {len(features)} features\n")
display(df.head())

n_missing = check_missing_values(df).sum()
n_duplicates = check_duplicates(df)

outlier_report = detect_outliers_iqr(df, features)
outlier_report.to_csv(os.path.join(RESULTS_DIR, "00_outlier_report.csv"), index=False)

print(f"\nMissing values (total): {n_missing}")
print(f"Duplicated rows: {n_duplicates}\n")
print("Top 8 features by outlier count (1.5*IQR rule):")
display(outlier_report.head(8))

**Findings.**
- No missing values and no duplicated rows. The dataset is clean. No imputation or
  de-duplication is needed before modeling.
- The features with the most flagged outliers are the standard-error (`_se`) variables
  and `area_worst`. These describe how much the nuclei vary within a single image, and
  the size of the largest nuclei. Both are expected to vary more in aggressive tumors.
  This is treated as a meaningful pattern to report, not an error to remove.

## Discussion

The dataset is clean and does not need cleaning before modeling. Outliers are kept,
not removed: an unusually large or irregular nucleus may reflect a more aggressive
tumor rather than a measurement error, and removing it would risk discarding real
biological signal.

This notebook only checks the data. The exploratory analysis in `03_EDA.ipynb` looks
at how the features actually behave.